In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import Layer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Rescaling, GlobalAveragePooling2D
from tensorflow.keras import layers, optimizers, callbacks

from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.applications import EfficientNetV2B2

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score
)

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)

import gradio as gr

In [ ]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    print(dirname)

In [ ]:
dataset_dir = r"/kaggle/input/datasets/jyotiyadav0000/imbalanced-garbage/TrashType_Image_Dataset"
image_size = (224,224)
batch_size = 32
seed = 42

#traininag dataset
train_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_dir,
    validation_split=0.2,
    subset="training",  # different from "validation"
    seed=seed,
    shuffle=True,
    image_size=image_size,
    batch_size=batch_size
)


In [ ]:
val_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_dir,
    validation_split=0.2,
    subset="validation",
    seed=seed,
    shuffle = True,
    image_size=image_size,
    batch_size=batch_size
)
val_class= val_ds.class_names



In [ ]:
# Get the total number of batches in the validation dataset
val_batches = tf.data.experimental.cardinality(val_ds)

# Split the validation dataset into two equal parts:
# First half becomes the test dataset
test_ds = val_ds.take(val_batches // 2)

# Second half remains as the validation dataset
val_dat = val_ds.skip(val_batches // 2)

# Optimize test dataset by caching and prefetching to improve performance
test_ds_eval = test_ds.cache().prefetch(tf.data.AUTOTUNE)

In [ ]:
print(train_ds.class_names)
print(val_class)
print(len(train_ds.class_names))


In [ ]:
plt.figure(figsize=(12, 12))

for images, labels in train_ds.take(1):
    for i in range(min(12, len(images))):

        ax = plt.subplot(4, 3, i + 1)

        plt.imshow(images[i].numpy().astype("uint8"))

        plt.title(class_names[labels[i]])

        plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
def count_distribution(dataset, class_names):
    total = 0
    counts = {name: 0 for name in class_names}

    for _, labels in dataset:
        for label in labels.numpy():
            class_name = class_names[label]
            counts[class_name] += 1
            total += 1

    if total == 0:
        print("⚠️ Dataset is empty. Cannot compute class distribution.")
        return counts  # Return raw zero counts (or empty if you prefer)

    for k in counts:
        counts[k] = round((counts[k] / total) * 100, 2)  # Convert to percentage

    return counts

In [ ]:
# Function to plot class distribution
def simple_bar_plot(dist, title):
    plt.bar(dist.keys(), dist.values(), color='cornflowerblue')
    plt.title(title)
    plt.ylabel('Percentage (%)')
    plt.xticks(rotation=45)
    plt.ylim(0, 100)
    plt.tight_layout()
    plt.show()

In [ ]:
class_names = train_ds.class_names

# Get class distributions
train_dist = count_distribution(train_ds, class_names)
val_dist = count_distribution(val_ds, class_names)
test_dist = count_distribution(test_ds, class_names)
overall_dist = {}
for k in class_names:
    overall_dist[k] = round((train_dist[k] + val_dist[k]) / 2, 2)

print(train_dist)
print(val_dist)
print(test_dist)
print(overall_dist)

In [ ]:
# Show visualizations
simple_bar_plot(train_dist, "Training Set Class Distribution (%)")
simple_bar_plot(val_dist, "Validation Set Class Distribution (%)")
simple_bar_plot(test_dist, "Test Set Class Distribution (%)")
simple_bar_plot(overall_dist, "Overall Class Distribution (%)")

In [ ]:
# Count class occurrences and prepare label list
class_counts = {i: 0 for i in range(len(class_names))}
all_labels = []

for images, labels in train_ds:
    for label in labels.numpy():
        class_counts[label] += 1
        all_labels.append(label)

# Compute class weights (index aligned)
class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(len(class_names)),
    y=all_labels
)

# Create dictionary mapping class index to weight
class_weights = {i: w for i, w in enumerate(class_weights_array)}

In [ ]:
print("Class Weights:")
for i, weight in class_weights.items():
    print(f"{class_names[i]}: {weight:.2f}")

In [ ]:
data_augmentation = Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.15),
    layers.RandomTranslation(
        height_factor=0.1,
        width_factor=0.1
    )
])

In [ ]:
base_model = EfficientNetV2B2(
    include_top=False,
    input_shape=(224,224,3),
    include_preprocessing=True,
    weights="imagenet"
)

base_model.trainable = True

for layer in base_model.layers[:100]:
    layer.trainable = False

print("Total layers:", len(base_model.layers))

In [ ]:
model = Sequential([
    layers.Input(shape=(224, 224, 3)),
    data_augmentation,
    base_model,
    GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(6, activation='softmax')
])

In [ ]:
model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
early = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=2,
    min_lr=1e-7,
    verbose=1
)

checkpoint = ModelCheckpoint(
    "best_model.keras",
    monitor="val_accuracy",
    save_best_only=True,
    mode="max",
    verbose=1
)

In [ ]:
# Number of epochs
epochs = 15

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=epochs,
    class_weight=class_weights,
    callbacks=[
        early,
        reduce_lr,
        checkpoint
    ]
)

In [ ]:
model.summary()
base_model.summary()

In [ ]:

acc = history.history['accuracy']          
val_acc = history.history['val_accuracy']  
loss = history.history['loss']             
val_loss = history.history['val_loss']     

epochs_range = range(len(acc))            

plt.figure(figsize=(10,8))                

plt.subplot(1,2,1)                         
plt.plot(epochs_range, acc, label='Training Accuracy')      
plt.plot(epochs_range, val_acc, label='Validation Accuracy') 
plt.legend(loc='lower right')              
plt.title('Training vs Validation Accuracy') 

plt.subplot(1,2,2)                         
plt.plot(epochs_range, loss, label='Training Loss')        
plt.plot(epochs_range, val_loss, label='Validation Loss')   
plt.legend(loc='upper right')              
plt.title('Training vs Validation Loss')   

plt.show()                                 

In [ ]:
loss, accuracy = model.evaluate(test_ds_eval)
print(f'Test accuracy is{accuracy:.4f}, Test loss is {loss:.4f}')

In [ ]:
    test_batches = list(test_ds_eval)

    # Extract true labels
    y_true = np.concatenate([y.numpy() for x, y in test_batches], axis=0)

    # Predict class probabilities
    y_pred_probs = model.predict(test_ds_eval)
    y_pred = np.argmax(y_pred_probs, axis=1)

    # Compute and display confusion matrix and classification report
    cm = confusion_matrix(y_true, y_pred)
    print("✅ Confusion Matrix:")
    print(cm)

    print("\n📊 Classification Report:")
    print(classification_report(y_true, y_pred))




In [ ]:
    # Additional Metrics
    precision = precision_score(y_true, y_pred, average='weighted')
    recall = recall_score(y_true, y_pred, average='weighted')
    f1 = f1_score(y_true, y_pred, average='weighted')
    
    print(f"\nPrecision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")

In [ ]:
plt.figure(figsize=(8,6))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
misclassified_images = []
misclassified_true = []
misclassified_pred = []

for images, labels in test_ds_eval:

    preds = model.predict(images, verbose=0)
    pred_labels = np.argmax(preds, axis=1)

    for i in range(len(labels)):
        if labels[i] != pred_labels[i]:

            misclassified_images.append(images[i].numpy())
            misclassified_true.append(labels[i].numpy())
            misclassified_pred.append(pred_labels[i])

In [ ]:
plt.figure(figsize=(15,10))

num_images = min(9, len(misclassified_images))

for i in range(num_images):

    plt.subplot(3,3,i+1)

    plt.imshow(misclassified_images[i].astype("uint8"))

    plt.title(
        f"True: {class_names[misclassified_true[i]]}\n"
        f"Pred: {class_names[misclassified_pred[i]]}"
    )

    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Sample Predictions on Test Images
class_names = train_ds.class_names

for images, labels in test_ds_eval.take(1):

    predictions = model.predict(images)

    pred_labels = tf.argmax(predictions, axis=1)

    plt.figure(figsize=(15,10))

for i in range(8):
    plt.subplot(2,4,i+1)

    plt.imshow(images[i].numpy().astype("uint8"))

    plt.title(
        f"T: {class_names[labels[i]]}\n"
        f"P: {class_names[pred_labels[i]]}"
    )

    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Save model in Keras format with architecture, weights, and training configuration
model.save('garbage_classifier.keras')

# Load Keras model
model = tf.keras.models.load_model('garbage_classifier.keras')

In [ ]:
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input

In [ ]:
def classify_image(img):

    img = img.resize((224, 224))

    img_array = np.array(img, dtype=np.float32)

    img_array = np.expand_dims(img_array, axis=0)

    prediction = model.predict(img_array, verbose=0)

    predicted_class_index = np.argmax(prediction)

    predicted_class_name = class_names[predicted_class_index]

    confidence = prediction[0][predicted_class_index]

    return f"Predicted: {predicted_class_name} (Confidence: {confidence:.2%})"

In [ ]:
iface = gr.Interface(
    fn=classify_image,  # Function to classify image using the trained model
    inputs=gr.Image(type="pil"),  # Accepts input as a PIL image
    outputs="text"  # Outputs prediction as text
)

# Launch the interface
iface.launch()  # Start the Gradio interface for user interaction